# Notebook 15 - Speech Plus Document Workflows

This notebook combines transcripts with pages, notes, or slides so answers can cite both what was said and where related document evidence lives.


## What you will learn

- how to join transcript spans with document chunks
- how to preserve two evidence channels at once
- how to produce grounded answers across speech and documents


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "15_speech_plus_document_workflows"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Pair transcript spans with pages

Meetings, lectures, and walkthroughs often reference visual material that lives in separate documents or slides.


In [ ]:
transcript = pd.DataFrame([
    {"segment_id": "s1", "timestamp": "00:00-00:08", "text": "Open the invoice summary on page one."},
    {"segment_id": "s2", "timestamp": "00:08-00:17", "text": "The total should be one thousand two hundred forty dollars."},
])
pages = pd.DataFrame([
    {"page_id": "p1", "page_note": "Invoice summary with total in bottom-right field."},
    {"page_id": "p2", "page_note": "Policy appendix."},
])
transcript, pages


## Step 2: Build a multimodal evidence bundle

The system answer should keep both timestamp evidence and page evidence instead of collapsing them into one vague citation.


In [ ]:
evidence_bundle = {
    "question": "What total did the speaker mention?",
    "answer": "$1,240.00",
    "evidence": [
        {"type": "audio", "span": "00:08-00:17", "quote": "one thousand two hundred forty dollars"},
        {"type": "document", "page_id": "p1", "field": "total", "box": "620,980,810,1020"},
    ],
}
print(json.dumps(evidence_bundle, indent=2))


## Step 3: Score dual-channel grounding

A good workflow checks whether both channels line up, not just whether one of them looks plausible.


In [ ]:
def dual_channel_success(audio_hit, document_hit):
    return int(audio_hit and document_hit)

checks = pd.DataFrame([
    {"example": "invoice total", "audio_hit": True, "document_hit": True},
    {"example": "policy clause", "audio_hit": True, "document_hit": False},
])
checks["dual_channel_success"] = checks.apply(lambda row: dual_channel_success(row.audio_hit, row.document_hit), axis=1)
checks


## Exercises and extensions

1. Add slide ids or screenshot ids as a third evidence channel.
1. Create a failure bucket for cases where the transcript is right but the cited page is wrong.
